In [1]:
import json
import random
from collections import Counter

RAW_TRAIN_PATH = "/kaggle/input/datasets/pradhyumnapawale/train-raw1/train_raw.jsonl"
GOLDEN_PATH = "/kaggle/input/datasets/pradhyumnapawale/golden-test/golden_test.jsonl"
CHAT_TRAIN_PATH = "/kaggle/working/train_chat.jsonl"

SYSTEM_PROMPT = (
    "You are a financial SMS parser. Extract transaction details into strict JSON with exactly 7 keys: "
    "is_transaction, type, amount, merchant, payment_mode, account_last4, date. "
    "If the SMS is not a completed money movement (e.g. OTP, failed txn, promo, balance alert, reminder), "
    "set is_transaction to false and all other 6 fields to null. Output ONLY valid JSON, no extra text."
)

# 1. Load datasets
with open(GOLDEN_PATH, "r", encoding="utf-8") as f:
    golden_sms = {json.loads(line)["sms"].strip().lower() for line in f}

with open(RAW_TRAIN_PATH, "r", encoding="utf-8") as f:
    raw_train = [json.loads(line) for line in f]

# 2. Leakage check & balance stats
clean_train = [x for x in raw_train if x["sms"].strip().lower() not in golden_sms]
print(f"Raw: {len(raw_train)} | Leaks removed: {len(raw_train) - len(clean_train)} | Clean: {len(clean_train)}")
print("is_transaction :", dict(Counter(x["expected"]["is_transaction"] for x in clean_train)))
print("type           :", dict(Counter(x["expected"]["type"] for x in clean_train)))
print("payment_mode   :", dict(Counter(x["expected"]["payment_mode"] for x in clean_train)))

# 3. Shuffle & save in 3-role chat format
random.seed(42)
random.shuffle(clean_train)

with open(CHAT_TRAIN_PATH, "w", encoding="utf-8") as f:
    for item in clean_train:
        record = {
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": item["sms"]},
                {"role": "assistant", "content": json.dumps(item["expected"])}
            ]
        }
        f.write(json.dumps(record) + "\n")

print(f"\nReady for Phase 4! Saved {len(clean_train)} chat examples to {CHAT_TRAIN_PATH}")

Raw: 444 | Leaks removed: 0 | Clean: 444
is_transaction : {True: 236, False: 208}
type           : {'debit': 180, None: 208, 'credit': 56}
payment_mode   : {'UPI': 137, None: 208, 'Card': 29, 'ATM': 10, 'NetBanking': 60}

Ready for Phase 4! Saved 444 chat examples to /kaggle/working/train_chat.jsonl


In [2]:
!pip install -q unsloth

In [3]:
import torch
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=512,   # SMS + JSON is short (~150 tokens), so 512 is plenty and fast
    dtype=None,           # Auto-detects float16 for T4
    load_in_4bit=True,
)

print(f"Loaded {MODEL_NAME} on {torch.cuda.get_device_name(0)}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.7: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Unsloth: Not splitting across GPUs; the model fits on cuda:0 with room to train. Pass device_map = "unsloth_balanced" to split it anyway.


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loaded unsloth/Qwen2.5-3B-Instruct-bnb-4bit on Tesla T4


In [4]:
import torch
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=512,   # SMS + JSON is short (~150 tokens), so 512 is plenty and fast
    dtype=None,           # Auto-detects float16 for T4
    load_in_4bit=True,
)

print(f"Loaded {MODEL_NAME} on {torch.cuda.get_device_name(0)}")

==((====))==  Unsloth 2026.11.7: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Unsloth: Not splitting across GPUs; the model fits on cuda:0 with room to train. Pass device_map = "unsloth_balanced" to split it anyway.


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loaded unsloth/Qwen2.5-3B-Instruct-bnb-4bit on Tesla T4


In [5]:
import json
import time

REQUIRED_KEYS = [
    "is_transaction", "type", "amount",
    "merchant", "payment_mode", "account_last4", "date"
]

def normalize_value(key, val):
    if val is None:
        return None
    if key == "amount":
        try:
            return round(float(val), 2)
        except (ValueError, TypeError):
            return val
    if isinstance(val, str):
        return val.strip().lower()
    return val

def evaluate_predictions(golden, predictions, label="MODEL"):
    total = len(golden)
    valid_json, schema_match, exact_match = 0, 0, 0
    field_correct = {k: 0 for k in REQUIRED_KEYS}

    for item, raw_pred in zip(golden, predictions):
        expected = item["expected"]
        try:
            pred = json.loads(raw_pred.strip())
            if not isinstance(pred, dict):
                continue
            valid_json += 1
        except json.JSONDecodeError:
            continue

        if set(pred.keys()) == set(REQUIRED_KEYS):
            schema_match += 1

        all_match = True
        for k in REQUIRED_KEYS:
            if normalize_value(k, expected.get(k)) == normalize_value(k, pred.get(k)):
                field_correct[k] += 1
            else:
                all_match = False
        if all_match:
            exact_match += 1

    print("=" * 48)
    print(f"{label} SCORECARD ({total} examples)")
    print("=" * 48)
    print(f"Strict JSON Validity : {valid_json}/{total} ({valid_json/total*100:.1f}%)")
    print(f"Exact Schema Match   : {schema_match}/{total} ({schema_match/total*100:.1f}%)")
    print(f"100% Exact Match     : {exact_match}/{total} ({exact_match/total*100:.1f}%)")
    print("-" * 48)
    for k in REQUIRED_KEYS:
        print(f"  {k:<15}: {field_correct[k]:>2}/{total} ({field_correct[k]/total*100:.1f}%)")
    print("=" * 48)

def run_inference(model, tokenizer, golden_data):
    FastLanguageModel.for_inference(model)
    preds = []
    t0 = time.time()

    for item in golden_data:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": item["sms"]},
        ]
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=160,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        # Slice out only the newly generated tokens
        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        preds.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())

    avg_ms = (time.time() - t0) / len(golden_data) * 1000
    return preds, avg_ms

with open(GOLDEN_PATH, "r", encoding="utf-8") as f:
    golden_data = [json.loads(line) for line in f]

base_preds, base_ms = run_inference(model, tokenizer, golden_data)
evaluate_predictions(golden_data, base_preds, label="BEFORE (BASE QWEN-2.5-3B)")
print(f"Avg inference time: {base_ms:.0f} ms / SMS")
print("\nSample raw output #1:\n", base_preds[0])

Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

BEFORE (BASE QWEN-2.5-3B) SCORECARD (30 examples)
Strict JSON Validity : 30/30 (100.0%)
Exact Schema Match   : 30/30 (100.0%)
100% Exact Match     : 5/30 (16.7%)
------------------------------------------------
  is_transaction : 28/30 (93.3%)
  type           : 18/30 (60.0%)
  amount         : 28/30 (93.3%)
  merchant       : 21/30 (70.0%)
  payment_mode   : 19/30 (63.3%)
  account_last4  : 13/30 (43.3%)
  date           :  8/30 (26.7%)
Avg inference time: 1465 ms / SMS

Sample raw output #1:
 {"is_transaction": true, "type": "debit", "amount": "245.50", "merchant": "UPI user A/C X482野", "payment_mode": null, "account_last4": "X4821", "date": "18Nov26"}


In [6]:
from datasets import load_dataset

# 1. Attach LoRA adapters (the "sticky notes" from Lesson 1)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

# 2. Load train_chat.jsonl and apply Qwen's chat template
dataset = load_dataset("json", data_files=CHAT_TRAIN_PATH, split="train")

def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = dataset.map(format_chat)
print(f"Prepared {len(dataset)} training examples.")

Unsloth 2026.11.7 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/444 [00:00<?, ? examples/s]

Prepared 444 training examples.


In [7]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_length=512,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,  # Effective batch size = 16
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=True,                      # Tesla T4 uses fp16 (bfloat16 = False)
        bf16=False,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="/kaggle/working/qwen_sms_lora",
        report_to="none",
    ),
)

trainer_stats = trainer.train()
print(f"\nTraining complete in {trainer_stats.metrics['train_runtime']:.1f} seconds!")

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/444 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=2):   0%|          | 0/444 [00:00<?, ? examples/s]

Unsloth: Packing is on by default - documents are packed into full rows with no cross-document attention. Set packing = False in SFTConfig (or UNSLOTH_DEFAULT_PACKING=0) to turn it off.


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 213 | Num Epochs = 3 | Total steps = 42
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: packed / padding-free rows run each transformer block uncompiled; only unpacked batches use the compiled blocks.
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,2.790709
20,1.213661
30,0.553956
40,0.462468


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/qwen_sms_lora/checkpoint-42/tokenizer_config.json.



Training complete in 350.6 seconds!


In [8]:
# 1. Run inference with the newly fine-tuned model
after_preds, after_ms = run_inference(model, tokenizer, golden_data)
evaluate_predictions(golden_data, after_preds, label="AFTER (DISTILLED QWEN-2.5-3B)")
print(f"Avg inference time: {after_ms:.0f} ms / SMS")
print("\nSample raw output #1:\n", after_preds[0])

# 2. Save the trained LoRA adapter (~120 MB)
model.save_pretrained("/kaggle/working/sms_lora_adapter")
tokenizer.save_pretrained("/kaggle/working/sms_lora_adapter")
print("\nSaved LoRA adapter to /kaggle/working/sms_lora_adapter")

Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

AFTER (DISTILLED QWEN-2.5-3B) SCORECARD (30 examples)
Strict JSON Validity : 30/30 (100.0%)
Exact Schema Match   : 30/30 (100.0%)
100% Exact Match     : 18/30 (60.0%)
------------------------------------------------
  is_transaction : 30/30 (100.0%)
  type           : 30/30 (100.0%)
  amount         : 30/30 (100.0%)
  merchant       : 28/30 (93.3%)
  payment_mode   : 23/30 (76.7%)
  account_last4  : 24/30 (80.0%)
  date           : 30/30 (100.0%)
Avg inference time: 1674 ms / SMS

Sample raw output #1:
 {"is_transaction": true, "type": "debit", "amount": 245.5, "merchant": "ZOMATO", "payment_mode": "UPI", "account_last4": "X4821", "date": "2026-11-18"}


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/sms_lora_adapter/tokenizer_config.json.



Saved LoRA adapter to /kaggle/working/sms_lora_adapter


In [9]:
import json

for i, (item, raw_pred) in enumerate(zip(golden_data, after_preds)):
    exp = item["expected"]
    pred = json.loads(raw_pred.strip())
    diffs = {}
    for k in REQUIRED_KEYS:
        if normalize_value(k, exp.get(k)) != normalize_value(k, pred.get(k)):
            diffs[k] = {"expected": exp.get(k), "got": pred.get(k)}
    if diffs:
        print(f"--- Mistake #{i+1} ---")
        print("SMS :", item["sms"])
        for k, v in diffs.items():
            print(f"  {k}: expected={v['expected']!r} | got={v['got']!r}")

--- Mistake #1 ---
SMS : Dear UPI user A/C X4821 debited by 245.50 on date 18Nov26 trf to ZOMATO Refno 632189451234. If not u? call 1800111109 -SBI
  account_last4: expected='4821' | got='X4821'
--- Mistake #2 ---
SMS : Sent Rs.120.00 From HDFC Bank A/C *9012 To SWIGGY INSTAMART On 15/11/26 Ref 631944201122. Not You? Call 18002586161
  payment_mode: expected='UPI' | got='NetBanking'
--- Mistake #4 ---
SMS : INR 50.00 debited from A/c no. XX3310 on 10-11-2026 09:14:22 via UPI/P2M/631499887766/SHARMA CHAIWALA. Avl Bal: INR 12,450.20 -Axis Bank
  payment_mode: expected='UPI' | got='UPI/P2M'
--- Mistake #5 ---
SMS : Sent Rs.899 from Kotak Bank AC X6109 to blinkit.payu@hdfcbank on 08-11-26.UPI Ref 631255443322. Not you? Call 18002740110.
  account_last4: expected='6109' | got='X6109'
--- Mistake #10 ---
SMS : Debited INR 199.00 from SBI A/c X4821 on 29Oct26 by UPI to SPOTIFY INDIA. Ref 630299887766.
  account_last4: expected='4821' | got='X4821'
--- Mistake #11 ---
SMS : Dear Customer, your

In [10]:
import json
import re
import random
from collections import Counter

RAW_TRAIN_PATH = "/kaggle/input/datasets/pradhyumnapawale/train-raw1/train_raw.jsonl"
GOLDEN_PATH = "/kaggle/input/datasets/pradhyumnapawale/golden-test/golden_test.jsonl"
CHAT_TRAIN_V2_PATH = "/kaggle/working/train_chat_v2.jsonl"

# Slightly tighter prompt so the model knows the exact allowed enums & 4-digit rule
SYSTEM_PROMPT_V2 = (
    "You are a financial SMS parser. Extract transaction details into strict JSON with exactly 7 keys: "
    "is_transaction, type, amount, merchant, payment_mode, account_last4, date. "
    "Rules: payment_mode must be ONLY one of 'UPI', 'Card', 'NetBanking', 'ATM', or null. "
    "account_last4 must be 4 digits only (no X or *). "
    "If not a completed money movement, set is_transaction to false and all other 6 fields to null. "
    "Output ONLY valid JSON, no extra text."
)

def clean_payment_mode(mode, sms):
    if mode is None:
        return None
    m = str(mode).strip().lower()
    sms_l = sms.lower()
    if "atm" in m or "cash withdrawal" in sms_l or "withdrawn from" in sms_l:
        return "ATM"
    if "upi" in m:
        return "UPI"
    if "card" in m or "debit card" in m or "credit card" in m:
        return "Card"
    if any(k in m for k in ["net", "imps", "neft", "rtgs", "sip", "emi", "auto"]):
        return "NetBanking"
    return mode  # fallback if already clean

def clean_account_last4(val):
    if val is None:
        return None
    digits = re.sub(r"\D", "", str(val))  # strip X, XX, *, spaces
    return digits[-4:] if len(digits) >= 4 else (digits or None)

with open(GOLDEN_PATH, "r", encoding="utf-8") as f:
    golden_sms = {json.loads(line)["sms"].strip().lower() for line in f}

with open(RAW_TRAIN_PATH, "r", encoding="utf-8") as f:
    raw_train = [json.loads(line) for line in f]

cleaned = []
fixed_acct, fixed_mode = 0, 0

for item in raw_train:
    if item["sms"].strip().lower() in golden_sms:
        continue
    exp = dict(item["expected"])
    if exp["is_transaction"]:
        new_acct = clean_account_last4(exp.get("account_last4"))
        if new_acct != exp.get("account_last4"):
            fixed_acct += 1
        exp["account_last4"] = new_acct

        new_mode = clean_payment_mode(exp.get("payment_mode"), item["sms"])
        if new_mode != exp.get("payment_mode"):
            fixed_mode += 1
        exp["payment_mode"] = new_mode

    cleaned.append({"sms": item["sms"], "expected": exp})

print(f"Fixed account_last4 in {fixed_acct} rows")
print(f"Fixed payment_mode  in {fixed_mode} rows")
print("Clean payment_mode distribution:", dict(Counter(x["expected"]["payment_mode"] for x in cleaned)))

random.seed(42)
random.shuffle(cleaned)

with open(CHAT_TRAIN_V2_PATH, "w", encoding="utf-8") as f:
    for item in cleaned:
        record = {
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT_V2},
                {"role": "user", "content": item["sms"]},
                {"role": "assistant", "content": json.dumps(item["expected"])}
            ]
        }
        f.write(json.dumps(record) + "\n")

print(f"\nSaved {len(cleaned)} cleaned examples to {CHAT_TRAIN_V2_PATH}")

Fixed account_last4 in 0 rows
Fixed payment_mode  in 2 rows
Clean payment_mode distribution: {'UPI': 137, None: 208, 'Card': 29, 'ATM': 12, 'NetBanking': 58}

Saved 444 cleaned examples to /kaggle/working/train_chat_v2.jsonl


In [11]:
import gc, torch
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from unsloth import FastLanguageModel

# Free VRAM from v1 model
del model, trainer
gc.collect()
torch.cuda.empty_cache()

# 1. Reload clean base 4-bit model & attach fresh LoRA adapters
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=512,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

# 2. Format cleaned v2 dataset
dataset_v2 = load_dataset("json", data_files=CHAT_TRAIN_V2_PATH, split="train")
dataset_v2 = dataset_v2.map(lambda ex: {
    "text": tokenizer.apply_chat_template(ex["messages"], tokenize=False, add_generation_prompt=False)
})

# 3. Train v2
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset_v2,
    args=SFTConfig(
        dataset_text_field="text",
        max_length=512,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=True,
        bf16=False,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="/kaggle/working/qwen_sms_lora_v2",
        report_to="none",
    ),
)
trainer.train()

# 4. Evaluate v2 (using SYSTEM_PROMPT_V2)
SYSTEM_PROMPT = SYSTEM_PROMPT_V2
v2_preds, v2_ms = run_inference(model, tokenizer, golden_data)
evaluate_predictions(golden_data, v2_preds, label="AFTER v2 (CLEANED DATA QWEN-2.5-3B)")
print(f"Avg inference time: {v2_ms:.0f} ms / SMS")

# 5. Save v2 adapter
model.save_pretrained("/kaggle/working/sms_lora_adapter_v2")
tokenizer.save_pretrained("/kaggle/working/sms_lora_adapter_v2")

==((====))==  Unsloth 2026.11.7: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Unsloth: Not splitting across GPUs; the model fits on cuda:0 with room to train. Pass device_map = "unsloth_balanced" to split it anyway.


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).


Generating train split: 0 examples [00:00, ? examples/s]

... (7 outputs left out)

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/qwen_sms_lora_v2/checkpoint-42/tokenizer_config.json.
Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please 

AFTER v2 (CLEANED DATA QWEN-2.5-3B) SCORECARD (30 examples)
Strict JSON Validity : 30/30 (100.0%)
Exact Schema Match   : 30/30 (100.0%)
100% Exact Match     : 22/30 (73.3%)
------------------------------------------------
  is_transaction : 29/30 (96.7%)
  type           : 29/30 (96.7%)
  amount         : 29/30 (96.7%)
  merchant       : 26/30 (86.7%)
  payment_mode   : 26/30 (86.7%)
  account_last4  : 29/30 (96.7%)
  date           : 28/30 (93.3%)
Avg inference time: 1665 ms / SMS


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/sms_lora_adapter_v2/tokenizer_config.json.


('/kaggle/working/sms_lora_adapter_v2/tokenizer_config.json',
 '/kaggle/working/sms_lora_adapter_v2/chat_template.jinja',
 '/kaggle/working/sms_lora_adapter_v2/tokenizer.json')

In [4]:
import json, time, torch
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from trl import SFTTrainer, SFTConfig

# 3. Train v3 (3 epochs, loss-masked, no packing)
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset_v3,
    args=SFTConfig(
        dataset_text_field="text",
        max_length=512,
        packing=False,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=True,
        bf16=False,
        logging_steps=20,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="/kaggle/working/qwen_sms_lora_v3",
        report_to="none",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
trainer.train()

# 4. Evaluate on Golden Test Set
def normalize_value(key, val):
    if val is None: return None
    if key == "amount":
        try: return round(float(val), 2)
        except: return val
    return val.strip().lower() if isinstance(val, str) else val

FastLanguageModel.for_inference(model)
preds, t0 = [], time.time()
for item in golden_data:
    prompt = tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT_V2}, {"role": "user", "content": item["sms"]}],
        tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=160, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    preds.append(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip())

avg_ms = (time.time() - t0) / len(golden_data) * 1000
total, valid_json, schema_match, exact_match = len(golden_data), 0, 0, 0
field_correct = {k: 0 for k in REQUIRED_KEYS}

for i, (item, raw_pred) in enumerate(zip(golden_data, preds)):
    exp = item["expected"]
    try:
        pred = json.loads(raw_pred)
        if not isinstance(pred, dict): continue
        valid_json += 1
    except:
        print(f"Mistake #{i+1}: Invalid JSON -> {raw_pred}")
        continue
    if set(pred.keys()) == set(REQUIRED_KEYS): schema_match += 1
    diffs = {}
    for k in REQUIRED_KEYS:
        if normalize_value(k, exp.get(k)) == normalize_value(k, pred.get(k)):
            field_correct[k] += 1
        else:
            diffs[k] = (exp.get(k), pred.get(k))
    if not diffs:
        exact_match += 1
    else:
        print(f"Mistake #{i+1} | SMS: {item['sms'][:75]}... | Diffs: {diffs}")

print("\n" + "=" * 48)
print(f"AFTER v3 (LOSS-MASKED, 3 EPOCHS) ({total} examples)")
print("=" * 48)
print(f"Strict JSON Validity : {valid_json}/{total} ({valid_json/total*100:.1f}%)")
print(f"Exact Schema Match   : {schema_match}/{total} ({schema_match/total*100:.1f}%)")
print(f"100% Exact Match     : {exact_match}/{total} ({exact_match/total*100:.1f}%)")
print("-" * 48)
for k in REQUIRED_KEYS:
    print(f"  {k:<15}: {field_correct[k]:>2}/{total} ({field_correct[k]/total*100:.1f}%)")
print("=" * 48)
print(f"Avg inference time: {avg_ms:.0f} ms / SMS")

# 5. Save final adapter
model.save_pretrained("/kaggle/working/sms_lora_adapter_v3")
tokenizer.save_pretrained("/kaggle/working/sms_lora_adapter_v3")
print("Saved final v3 adapter to /kaggle/working/sms_lora_adapter_v3")

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/444 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/444 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 444 | Num Epochs = 3 | Total steps = 84
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: packed / padding-free rows run each transformer block uncompiled; only unpacked batches use the compiled blocks.
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
20,0.121173
40,0.011693
60,0.006041
80,0.003320


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/qwen_sms_lora_v3/checkpoint-84/tokenizer_config.json.
Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please 

Mistake #2 | SMS: Sent Rs.120.00 From HDFC Bank A/C *9012 To SWIGGY INSTAMART On 15/11/26 Ref... | Diffs: {'payment_mode': ('UPI', 'Card')}
Mistake #18 | SMS: Rs.5,000.00 withdrawn from SBI ATM using Card X7890 on 11Nov26 at VISHNUPUR... | Diffs: {'merchant': ('SBI ATM', 'VISHNUPURI NANDED')}
Mistake #20 | SMS: Cash withdrawal of Rs 2,000.00 made on HDFC Bank Debit Card ending 9901 at ... | Diffs: {'merchant': (None, 'ATM')}

AFTER v3 (LOSS-MASKED, 3 EPOCHS) (30 examples)
Strict JSON Validity : 30/30 (100.0%)
Exact Schema Match   : 30/30 (100.0%)
100% Exact Match     : 27/30 (90.0%)
------------------------------------------------
  is_transaction : 30/30 (100.0%)
  type           : 30/30 (100.0%)
  amount         : 30/30 (100.0%)
  merchant       : 28/30 (93.3%)
  payment_mode   : 29/30 (96.7%)
  account_last4  : 30/30 (100.0%)
  date           : 30/30 (100.0%)
Avg inference time: 1659 ms / SMS


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/sms_lora_adapter_v3/tokenizer_config.json.


Saved final v3 adapter to /kaggle/working/sms_lora_adapter_v3


In [6]:
import shutil
from IPython.display import FileLink

shutil.make_archive("/kaggle/working/sms_lora_adapter_v3", "zip", "/kaggle/working/sms_lora_adapter_v3")
FileLink("/kaggle/working/sms_lora_adapter_v3.zip")

/kaggle/working/sms_lora_adapter_v3.zip

In [7]:
import json
import time
import torch
from unsloth import FastLanguageModel

# Ensure inference mode is active (2x faster generation)
FastLanguageModel.for_inference(model)

SYSTEM_PROMPT_V2 = (
    "You are a financial SMS parser. Extract transaction details into strict JSON with exactly 7 keys: "
    "is_transaction, type, amount, merchant, payment_mode, account_last4, date. "
    "Rules: payment_mode must be ONLY one of 'UPI', 'Card', 'NetBanking', 'ATM', or null. "
    "account_last4 must be 4 digits only (no X or *). "
    "If not a completed money movement, set is_transaction to false and all other 6 fields to null. "
    "Output ONLY valid JSON, no extra text."
)

def parse_sms(sms_text: str) -> dict:
    """Runs the distilled v3 model on a single SMS and returns parsed JSON."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_V2},
        {"role": "user", "content": sms_text.strip()},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    t0 = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=160,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    latency_ms = (time.time() - t0) * 1000

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    raw_output = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    print(f"Latency: {latency_ms:.0f} ms")
    try:
        parsed = json.loads(raw_output)
        print(json.dumps(parsed, indent=2))
        return parsed
    except json.JSONDecodeError:
        print("Warning: Model returned invalid JSON:\n", raw_output)
        return {"raw_output": raw_output}

# --- TEST IT LIVE ---
# Paste any real SMS from your phone inside the triple quotes:
my_sms = """
A/C X4821 debited by Rs 349.00 on 20Nov26 transfer to JIO PREPAID Ref No 632411223344. Avl Bal Rs 9,420.50 -SBI
"""

result = parse_sms(my_sms)

Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Latency: 1814 ms
{
  "is_transaction": true,
  "type": "debit",
  "amount": 349.0,
  "merchant": "JIO PREPAID",
  "payment_mode": "Card",
  "account_last4": "4821",
  "date": "2026-11-20"
}


In [8]:
import os
from IPython.display import FileLink

# 1. Check if the saved adapter folder actually exists
print("Files in /kaggle/working:", os.listdir("/kaggle/working"))

# 2. Zip it cleanly into the current working directory
!zip -r sms_lora_adapter_v3.zip /kaggle/working/sms_lora_adapter_v3

# 3. Verify the zip was created and check its size
!ls -lh sms_lora_adapter_v3.zip

# 4. Relative path works much better with FileLink
FileLink("sms_lora_adapter_v3.zip")

Files in /kaggle/working: ['train_chat_v2.jsonl', 'huggingface_tokenizers_cache', 'sms_lora_adapter_v3', '.virtual_documents', 'sms_lora_adapter_v3.zip', 'unsloth_compiled_cache', 'qwen_sms_lora_v3']
  adding: kaggle/working/sms_lora_adapter_v3/ (stored 0%)
  adding: kaggle/working/sms_lora_adapter_v3/chat_template.jinja (deflated 71%)
  adding: kaggle/working/sms_lora_adapter_v3/adapter_model.safetensors (deflated 8%)
  adding: kaggle/working/sms_lora_adapter_v3/tokenizer.json (deflated 81%)
  adding: kaggle/working/sms_lora_adapter_v3/adapter_config.json (deflated 60%)
  adding: kaggle/working/sms_lora_adapter_v3/tokenizer_config.json (deflated 89%)
  adding: kaggle/working/sms_lora_adapter_v3/README.md (deflated 65%)
-rw-r--r-- 1 root root 215M Nov 20 10:30 sms_lora_adapter_v3.zip


/kaggle/working/sms_lora_adapter_v3.zip